# **Self-Improving In-Context Learning**

This notebook demonstrates a method for improving in-context learning (ICL) performance by optimizing prompt embeddings. The method designs a self-supervised proxy that returns the model's in-context task confidence as a bounded scalar value, then maximizes this proxy with respect to the input embeddings using zeroth-order optimization. No white-box access, training, auxiliary data, or learned parameters are required.

### Requirements

- A GPU with at least 24 GB VRAM (for Llama 3.1-8B in FP16)
- Access to `meta-llama/Llama-3.1-8B` on HuggingFace

```bash
pip install torch transformers accelerate
```

## Setup

In [1]:
import json
import os
import logging

import torch

from sicl import (
    init_model,
    set_seed,
    find_token_spans_from_sample,
    render_prompt,
    get_template_config_from_sample,
    OptimizationConfig,
    optimize_sample,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
set_seed(42)

In [2]:
model, tokenizer, embed_layer = init_model({
    "pretrained_model_name_or_path": "meta-llama/Llama-3.1-8B",
    "torch_dtype": torch.float16,
    "device_map": "auto",
})

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Load the ICLEval benchmark

ICLEval contains 12 tasks (2,040 samples total) spanning format recognition, ordering, deduplication, navigation, and relational reasoning. Each sample provides few-shot exemplars followed by a query.

In [3]:
DATA_DIR = "data/tasks_data"
benchmark = {}
for fname in sorted(os.listdir(DATA_DIR)):
    if fname.endswith(".json") and not fname.startswith("token_"):
        task_name = fname.replace(".json", "")
        with open(os.path.join(DATA_DIR, fname)) as f:
            benchmark[task_name] = json.load(f)

print(f"Loaded {sum(len(v) for v in benchmark.values())} samples across {len(benchmark)} tasks")
for name, samples in benchmark.items():
    print(f"  {name}: {len(samples)} samples")

Loaded 2040 samples across 12 tasks
  count_and_navigation: 120 samples
  deduplication: 300 samples
  dictionary_search: 190 samples
  duplication_check: 300 samples
  format_check: 120 samples
  format_cloning: 100 samples
  format_conversion: 120 samples
  list_mapping: 250 samples
  order_adjustment: 240 samples
  order_check: 100 samples
  relation_analysis: 100 samples
  string_completion: 100 samples


## Find a sample the model gets wrong

We select a sample from `format_check` where the model's baseline prediction is incorrect. The format classification task has single-token labels (table, csv, tuple, xml, yaml, jsonl), which provides strong proxy signal for optimization.

In [14]:
task_name = "format_check"

# Iterate to find a sample the model answers incorrectly at baseline.
# Once found, hardcode sample_idx below and skip this loop on future runs.
sample_idx = None
for idx, s in enumerate(benchmark[task_name]):
    cfg = get_template_config_from_sample(s)
    p = render_prompt(s, cfg)
    ids = tokenizer.encode(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip().lower()
    if s["label"].lower() not in pred:
        sample_idx = idx
        print(f"Found failing sample: idx={idx}, uid={s['uid']}, label={s['label']}, pred='{pred}'")
        break

assert sample_idx is not None, "All samples answered correctly — try a different task."
sample = benchmark[task_name][sample_idx]

Found failing sample: idx=0, uid=0, label=table, pred=''


In [15]:
config = get_template_config_from_sample(sample)
prompt = render_prompt(sample, config)

print(f"Task: {sample['task']} ({sample['task_type']})")
print(f"Ground truth: {sample['label']}")
print(f"\nPrompt:\n{prompt}")

Task: format_convert (normal)
Ground truth: table

Prompt:
Input:
|Index|name|age|city|
|---|---|---|---|
|1|Ava Hill|31|Portland|
Output: table

Input:
Index,name,age,city
1,David Wilson,29,Boston
Output: csv

Input:
(Landon Smith, age, 31)
(Landon Smith, city, New Orleans)
Output: tuple

Input:
<person>
  <name>Alexander Harris</name>
  <age>33</age>
  <city>Philadelphia</city>
</person>
Output: xml

Input:
person:
  name: Charlotte Adams
  age: 35
  city: Seattle
Output: yaml

Input:
{"name": "Grace Morgan", "age": 30, "city": "St. Louis"}
Output: jsonl

Input:
|Index|name|age|city|
|---|---|---|---|
|1|Brooklyn Wilson|35|Tampa|
Output: 


## Baseline prediction

Generate from the original text prompt to confirm the model answers incorrectly.

In [16]:
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

baseline_answer = tokenizer.decode(output[0, input_ids.shape[1]:], skip_special_tokens=True)
print(f"Model prediction: {baseline_answer.strip()}")
print(f"Ground truth:     {sample['label']}")
print(f"Correct:          {sample['label'].lower() in baseline_answer.lower()}")

Model prediction: 
Ground truth:     table
Correct:          False


## Optimize prompt embeddings

The optimization maximizes an ICL confidence proxy with respect to the prompt's embedding representation. The proxy combines three signals:

$$f = \alpha \cdot \hat{C} + \beta \cdot \hat{R} + \gamma \cdot G$$

- $\hat{C}$: Per-exemplar confidence — geometric mean of label token probabilities, averaged across exemplars
- $\hat{R}$: Pooled robustness — 10th percentile of all label token probabilities
- $G$: Information gain — positive confidence changes between consecutive exemplars

The gradient is estimated via finite differences: sample $N$ random noise directions, evaluate the proxy at each perturbation, and compute:

$$\nabla f \approx \frac{1}{N \mu} \sum_{i=1}^{N} \left[ f(x + \mu z_i) - f(x) \right] z_i$$

A cosine similarity projection prevents the optimized embeddings from drifting too far from their original semantics.

In [ ]:
opt_config = OptimizationConfig()

optimized_embeddings, stats = optimize_sample(
    sample=sample,
    model=model,
    tokenizer=tokenizer,
    embed_layer=embed_layer,
    config=opt_config,
    verbose=True,
)

  Iteration 1/250 — Proxy: 0.116211, Cos Sim: 0.998047, SNR: 0.0036
  Iteration 2/250 — Proxy: 0.118408, Cos Sim: 0.996582, SNR: 0.0696
  Iteration 3/250 — Proxy: 0.121155, Cos Sim: 0.995605, SNR: 0.2629
  Iteration 4/250 — Proxy: 0.124390, Cos Sim: 0.994629, SNR: 0.3687
  Iteration 5/250 — Proxy: 0.127808, Cos Sim: 0.993164, SNR: 0.1180
  Iteration 6/250 — Proxy: 0.130981, Cos Sim: 0.992188, SNR: 0.0433
  Iteration 7/250 — Proxy: 0.134399, Cos Sim: 0.990723, SNR: 0.6016
  Iteration 8/250 — Proxy: 0.136963, Cos Sim: 0.989746, SNR: 0.4641
  Iteration 9/250 — Proxy: 0.139648, Cos Sim: 0.988281, SNR: 0.3586
  Iteration 10/250 — Proxy: 0.143555, Cos Sim: 0.987305, SNR: 0.7627
  Iteration 11/250 — Proxy: 0.145874, Cos Sim: 0.986328, SNR: 0.7681
  Iteration 12/250 — Proxy: 0.149170, Cos Sim: 0.984863, SNR: 0.2278
  Iteration 13/250 — Proxy: 0.152344, Cos Sim: 0.983887, SNR: 1.0312
  Iteration 14/250 — Proxy: 0.154175, Cos Sim: 0.982422, SNR: 0.8589
  Iteration 15/250 — Proxy: 0.155518, Cos S

In [8]:
print(f"Initial proxy:  {stats['initial_proxy']:.4f}")
print(f"Final proxy:    {stats['final_proxy']:.4f}")
print(f"Improvement:    {stats['proxy_improvement']:.4f}")
print(f"Best iteration: {stats['best_iteration']}/{stats['iterations_run']}")
print(f"Cosine sim:     {stats['final_cosine_similarity']:.4f}")

Initial proxy:  0.1162
Final proxy:    0.3574
Improvement:    0.2412
Best iteration: 236/250
Cosine sim:     0.7842


## **Verify:** Generate from optimized embeddings

Feed the optimized embeddings directly into the model and generate. The model should now predict the correct answer.

In [ ]:
with torch.no_grad():
    output = model.generate(
        inputs_embeds=optimized_embeddings.unsqueeze(0),
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

optimized_answer = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"Before optimization: {baseline_answer.strip()}")
print(f"After optimization:  {optimized_answer.strip()}")
print(f"Ground truth:        {sample['label']}")

Before optimization: 
After optimization:  table
Ground truth:        table
